# Phase 3 &mdash; Gold Layer

The Gold stage turns the cleaned Silver table into three business-ready, model-ready Delta tables: `gold_loans_train`, `gold_loans_val`, and `gold_loans_test`. These are the tables the modeling notebook reads from.

### What happens here

1. **Time-based split on `issue_d`** &mdash; mirrors our Phase 1 split so results stay comparable.
    - **Train:** first 80% of 2014&ndash;2016 loans (chronological)
    - **Validation:** remaining 20% of 2014&ndash;2016 loans
    - **Test:** *all* of 2017 (held out completely)

    Splitting by date instead of at random is essential for this problem: if we sampled randomly, the model would see loans from the future during training and the validation AUC would be misleadingly optimistic. The 2017 test set is our out-of-time holdout.

2. **Stratified downsample of Train/Val** &mdash; we sample 200,000 train rows and 50,000 validation rows while preserving the positive-class ratio. Reasons:
    - Keeps the dataset small enough to fit comfortably in the modeling notebook's memory.
    - Training time drops from minutes to seconds, which lets us iterate.
    - The charge-off rate is preserved exactly, so performance metrics stay meaningful.

    **The test set is kept at full size.** Reported test metrics therefore use every 2017 loan, making them directly comparable to Phase 2.

3. Imputation, encoding, and scaling are **deferred** to notebook 04. The same transformer gets fit on train and applied to val/test there, which is the only way to prevent leakage through the preprocessing step.

In [0]:
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder.appName("phase3-gold")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE = "workspace.default.silver_loans"
TRAIN_TABLE  = "workspace.default.gold_loans_train"
VAL_TABLE    = "workspace.default.gold_loans_val"
TEST_TABLE   = "workspace.default.gold_loans_test"

TRAIN_SAMPLE_N = 200_000
VAL_SAMPLE_N   = 50_000
SEED = 42

In [0]:
silver = spark.table(SILVER_TABLE).withColumn("issue_year", F.year("issue_d"))
print("Rows per issue year:")
silver.groupBy("issue_year").count().orderBy("issue_year").show()

Rows per issue year:
+----------+------+
|issue_year| count|
+----------+------+
|      2014|235616|
|      2015|402818|
|      2016|403032|
|      2017|314212|
+----------+------+



## Step 1 &mdash; Time-based split

Test is all of 2017. Train and validation come from 2014&ndash;2016, split chronologically using `percent_rank()` over `issue_d`. The first 80% of the ordered rows go to train, the remaining 20% go to validation. This matches how `src/04_data_splitting.py` split the data in Phase 1.

In [0]:
test_df = silver.filter(F.col("issue_year") == 2017)

trainval = silver.filter(F.col("issue_year").isin(2014, 2015, 2016))
w = Window.orderBy("issue_d")
trainval = trainval.withColumn("_rank", F.percent_rank().over(w))

train_df = trainval.filter(F.col("_rank") < 0.80).drop("_rank")
val_df   = trainval.filter(F.col("_rank") >= 0.80).drop("_rank")

print(f"Train (full):  {train_df.count():,} rows")
print(f"Val   (full):  {val_df.count():,} rows")
print(f"Test  (2017):  {test_df.count():,} rows")

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Train (full):  854,167 rows
Val   (full):  187,299 rows
Test  (2017):  314,212 rows


## Step 2 &mdash; Stratified sample of Train and Val

`sampleBy` draws a fraction of rows from each class, preserving the positive rate. The test set is left at full size.

In [0]:
def stratified_sample(df, target_n, label_col="charged_off", seed=SEED):
    """Sample about `target_n` rows while preserving the class ratio of `label_col`."""
    total = df.count()
    if total <= target_n:
        return df
    frac = target_n / total
    return df.stat.sampleBy(label_col, fractions={0: frac, 1: frac}, seed=seed)

train_sample = stratified_sample(train_df, TRAIN_SAMPLE_N)
val_sample   = stratified_sample(val_df,   VAL_SAMPLE_N)

for name, d in [("Train sample", train_sample),
                ("Val sample",   val_sample),
                ("Test (full)",  test_df)]:
    n = d.count()
    pos = d.filter(F.col("charged_off") == 1).count()
    print(f"{name:<14} {n:>8,} rows   |  charge-off rate {pos / n:.1%}")

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Train sample    200,796 rows   |  charge-off rate 19.3%
Val sample       50,260 rows   |  charge-off rate 19.6%
Test (full)     314,212 rows   |  charge-off rate 21.0%


## Step 3 &mdash; Write the three Gold tables

In [0]:
def write_delta(df, table):
    (
        df.drop("issue_year").write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    print(f"Wrote {table}")

write_delta(train_sample, TRAIN_TABLE)
write_delta(val_sample,   VAL_TABLE)
write_delta(test_df,      TEST_TABLE)

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Wrote workspace.default.gold_loans_train
Wrote workspace.default.gold_loans_val
Wrote workspace.default.gold_loans_test


## Step 4 &mdash; Quick sanity check

In [0]:
for t in [TRAIN_TABLE, VAL_TABLE, TEST_TABLE]:
    print(f"\n=== {t} ===")
    spark.table(t).limit(1).toPandas().T.head(15)


=== workspace.default.gold_loans_train ===

=== workspace.default.gold_loans_val ===

=== workspace.default.gold_loans_test ===
